In [1]:
from ingest import load_and_flatten_data
from rag import build_index, ask_assistant

documents = load_and_flatten_data()

print(f"Successfully processed {len(documents)} documents.")
print("\nSample document:")
documents[0]

Successfully processed 19 documents.

Sample document:


{'section': 'Origins',
 'question': "Who murdered Bruce Wayne's parents?",
 'text': 'Joe Chill is the mugger who murdered Thomas and Martha Wayne in Crime Alley, an event that drove Bruce to become Batman.',
 'category': 'Batman'}

In [2]:
# Build both indexes
minsearch_client = build_index(documents, backend="minsearch")

es_client = build_index(documents, backend="elasticsearch")

In [9]:
test_queries = [
    "Who is the street mugger responsible for the murder of Bruce's parents?",
    "What independent superhero identity did Dick Grayson take on when he grew up?",
    "What was the Joker's supposed profession according to theories about his past?",
    "What is the name of the massive secret base located beneath Wayne Manor?",
    "What is the name of the psychiatric hospital in Gotham where criminally insane villains are locked up?",
    "Who is Diana's mother and the queen of Themyscira?",
    "What is the magical weapon forged by Hephaestus that forces anyone bound by it to tell the truth?",
    "Who is the archaeologist villain cursed with a human-cheetah hybrid form?",
    "What was Superman's birth name on his home planet before it was destroyed?",
    "What radioactive fragment can drain Superman's powers and kill him?",
    # Hybrid / Multi-part questions combining two concepts:
    "Who murdered Bruce Wayne's parents and who is the first Robin?",
    "Where is Wonder Woman from and what magical weapon does she use that forces truth?",
    "What is Superman's birth name and what radioactive fragment is his primary weakness?"
]

print(f"Total of {len(test_queries)} custom test queries prepared.\n")

Total of 13 custom test queries prepared.



In [10]:
# Loop to test both search engines
for q in test_queries:
    # 1. Minsearch Pipeline
    min_answer = ask_assistant(q, minsearch_client, backend="minsearch")
    
    # 2. Elasticsearch Pipeline
    es_answer = ask_assistant(q, es_client, backend="elasticsearch")
    
    print(f" Question: {q}")
    print(f"  [Minsearch]     : {min_answer.strip()}")
    print(f"  [Elasticsearch] : {es_answer.strip()}")
    print("-" * 60)

 Question: Who is the street mugger responsible for the murder of Bruce's parents?
  [Minsearch]     : Joe Chill is the street mugger responsible for the murder of Bruce's parents.
  [Elasticsearch] : The street mugger responsible for the murder of Bruce's parents is Joe Chill.
------------------------------------------------------------
 Question: What independent superhero identity did Dick Grayson take on when he grew up?
  [Minsearch]     : Dick Grayson's independent superhero identity after growing up is Nightwing.
  [Elasticsearch] : Dick Grayson took on the independent superhero identity of Nightwing when he grew up.
------------------------------------------------------------
 Question: What was the Joker's supposed profession according to theories about his past?
  [Minsearch]     : According to the given context, the Joker was supposedly a failed stand-up comedian.
  [Elasticsearch] : The Joker's supposed profession according to theories about his past is a failed stand-up co

In [11]:
# Advanced test queries to see the differences (especially BM25 scoring vs in-memory simple search)
test_queries = [
    # Typo test (Minsearch and Elasticsearch might handle these kinds of errors differently)
    "Who is Joe Chil who murdered Thomas Wayne?",
    
    # Scattered keywords and long context test (different word order)
    "parents of Bruce Wayne mugger shooter name",
    
    # Semantic / indirect search test
    "Which psychiatric facility holds Gotham's most dangerous inmates?",
    
    # Tricky questions containing distractors
    "Who trained or raised Diana and rules the island of Amazons?"
]

print(f"Total of {len(test_queries)} advanced test queries prepared.\n")

Total of 4 advanced test queries prepared.



In [15]:
for q in test_queries:
    min_answer = ask_assistant(q, minsearch_client, backend="minsearch")
    es_answer = ask_assistant(q, es_client, backend="elasticsearch")
    
    print(f" Question: {q}")
    print(f"  [Minsearch]     : {min_answer.strip()}")
    print(f"  [Elasticsearch] : {es_answer.strip()}")
    print("-" * 60)

 Question: Who is Joe Chil who murdered Thomas Wayne?
  [Minsearch]     : Joe Chill is the mugger who murdered Thomas and Martha Wayne in Crime Alley, an event that drove Bruce to become Batman.
  [Elasticsearch] : Joe Chill murdered Thomas Wayne.
------------------------------------------------------------
 Question: parents of Bruce Wayne mugger shooter name
  [Minsearch]     : I don't have that information.

The provided context does not mention the name of the shooter of Bruce Wayne's mugger Joe Chill.
  [Elasticsearch] : I don't have that information.
------------------------------------------------------------
 Question: Which psychiatric facility holds Gotham's most dangerous inmates?
  [Minsearch]     : Arkham Asylum.
  [Elasticsearch] : Arkham Asylum is the psychiatric hospital and prison where many of Batman's most dangerous foes are incarcerated.
------------------------------------------------------------
 Question: Who trained or raised Diana and rules the island of Amazon